In [1]:
import sqlite3
import pandas as pd
import numpy as np

# 1. Initialize In-Memory SQLite Database
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# 2. Construct Table Schema
cur.execute("""
CREATE TABLE raw_patients (
    patient_id INTEGER PRIMARY KEY,
    full_name TEXT,
    gender_input TEXT,
    height_in REAL,
    weight_lbs REAL
);
""")

# 3. Seed Clinical Intake Data
raw_data = [
    (1, "John Doe", "  M  ", 71.2, 175.0),
    (2, "Jane Smith", "female", 64.5, 130.5),
    (3, "Mark Lee", "m", None, 160.2),
    (4, "Ana Cruz", "fem@le", 62.0, None),
    (5, "Bob Santos", "Male", 70.0, 185.0),
    (6, "Carla Dizon", "f ", 65.5, 142.0),
    (7, "David Reyes", "MALE", 195.0, 190.0),
    (8, "Elena Gomez", "female", 63.2, 128.0),
    (9, "Jane Smith", "female", 64.5, 130.5),
    (10, "Felix Tan", "m", 68.0, 168.0)
]

cur.executemany(
    "INSERT INTO raw_patients VALUES (?, ?, ?, ?, ?)",
    raw_data
)
conn.commit()

In [2]:
df_raw = pd.read_sql_query("SELECT * FROM raw_patients", conn)

df_raw

,patient_id,full_name,gender_input,height_in,weight_lbs
0,1,John Doe,M,71.2,175.0
1,2,Jane Smith,female,64.5,130.5
2,3,Mark Lee,m,NaN,160.2
3,4,Ana Cruz,fem@le,62.0,NaN
4,5,Bob Santos,Male,70.0,185.0
5,6,Carla Dizon,f,65.5,142.0
6,7,David Reyes,MALE,195.0,190.0
7,8,Elena Gomez,female,63.2,128.0
8,9,Jane Smith,female,64.5,130.5
9,10,Felix Tan,m,68.0,168.0


In [3]:
missing_counts = df_raw.isnull().sum()
duplicate_count = df_raw.duplicated().sum()

print("Missing values per column:")
print(missing_counts)

print("Duplicate rows:", duplicate_count)

Missing values per column:
patient_id      0
full_name       0
gender_input    0
height_in       1
weight_lbs      1
dtype: int64
Duplicate rows: 0


In [4]:
df_raw["gender_input"] = df_raw["gender_input"].str.strip().str.lower()

df_raw["gender_input"] = df_raw["gender_input"].replace({
    "m": "male",
    "f": "female",
    "fem@le": "female"
})

In [5]:
df_raw["height_in"] = df_raw["height_in"].fillna(df_raw["height_in"].median())
df_raw["weight_lbs"] = df_raw["weight_lbs"].fillna(df_raw["weight_lbs"].mean())

In [6]:
df_raw = df_raw.drop(columns=["patient_id"])
df_raw = df_raw.drop_duplicates()

df_raw = df_raw.rename(columns={
    "full_name": "Name",
    "gender_input": "Gender",
    "height_in": "Height(Inches)",
    "weight_lbs": "Weight(Pounds)"
})

df_raw.to_csv("heights_weights_dirty_v2.csv", index=False)

df_raw

,Name,Gender,Height(Inches),Weight(Pounds)
0,John Doe,male,71.2,175.000000
1,Jane Smith,female,64.5,130.500000
2,Mark Lee,male,65.5,160.200000
3,Ana Cruz,female,62.0,156.577778
4,Bob Santos,male,70.0,185.000000
5,Carla Dizon,female,65.5,142.000000
6,David Reyes,male,195.0,190.000000
7,Elena Gomez,female,63.2,128.000000
9,Felix Tan,male,68.0,168.000000
